# Tutorial 10: Chemical Format Conversion and Metadata Handling

Round-trip molecules between common chemical formats while preserving metadata fields.


## Objectives

- Load structure-data files (SDF) that contain rich metadata.
- Convert the records into pandas DataFrames for analysis.
- Export SMILES and SDF files with selected properties.


In [ ]:
from rdkit import Chem
from rdkit.Chem import PandasTools
import pandas as pd
from io import StringIO


## Read SDF Data

The snippet below emulates reading from disk by loading a multi-record SDF string.


In [ ]:
sdf_block = """
 Mrv2108 07152116512D

 6  5  0  0  0  0            999 V2000
   1.2990   -0.7500    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   0.0000   -1.5000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
  -1.2990   -0.7500    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
  -1.2990    0.7500    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   0.0000    1.5000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   1.2990    0.7500    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
 1  2  2  0  0  0  0
 2  3  1  0  0  0  0
 3  4  2  0  0  0  0
 4  5  1  0  0  0  0
 5  6  2  0  0  0  0
 6  1  1  0  0  0  0
M  END
>  <Name>
Benzene

>  <Source>
Example

$$$$
""".strip()
supplier = Chem.SDMolSupplier()
supplier.SetData(sdf_block, sanitize=True)
molecules = [mol for mol in supplier if mol is not None]
len(molecules)


## Convert to a DataFrame

`PandasTools.LoadSDF` retains all metadata fields, making downstream analytics straightforward.


In [ ]:
sdf_buffer = StringIO(sdf_block)
df = PandasTools.LoadSDF(sdf_buffer, smilesName='smiles', molColName='ROMol')
df


## Write SMILES and SDF Outputs

Export the curated data to SMILES or SDF files. String buffers allow inspection without touching disk.


In [ ]:
smiles_buffer = StringIO()
PandasTools.WriteSmi(df, smiles_buffer, molColName='ROMol', includeHeader=True, idName='Name')
smiles_buffer.getvalue()


In [ ]:
sdf_writer = Chem.SDWriter()
sdf_output = StringIO()
sdf_writer.SetOutputStream(sdf_output)
for mol in molecules:
    mol.SetProp('Processed', 'True')
    sdf_writer.write(mol)
sdf_writer.close()
sdf_output.getvalue().splitlines()[:10]
